In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install ultralytics opencv-python anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 37.7 MB/s eta 0:00:00


In [3]:
# ============================================================
# 13_llm_intent_analysis.ipynb
#
# Capa de análisis de intención con LLM (Claude) sobre el
# output del pipeline de detección de armas + HPE.
#
# Flujo por clip:
#   1. Detector de armas (Modelo B) -> frames con arma
#   2. YOLOv8-pose -> keypoints + clasificación AIMING/HOLDING/NEUTRAL
#   3. Filtro temporal -> threat_confirmed (bool)
#   4. [NUEVO] Resumen estructurado del clip -> Claude API
#      -> nivel de alerta + interpretación narrativa
#
# El LLM se invoca UNA VEZ por clip, al final del procesamiento,
# con el contexto completo acumulado.
#
# Opciones A+B:
#   A - Interpretación narrativa de las métricas calculadas
#   B - Razonamiento sobre el patrón temporal de poses
# ============================================================

import os
import shutil
import math
import json
import csv
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import anthropic
from ultralytics import YOLO

print('OK Imports')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
OK Imports


In [4]:
# ============================================================
# CONFIG
# ============================================================

# Rutas
POS_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt'
NEG_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt'
OUT_DIR  = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/llm_intent'

WEAPON_DET_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'
POSE_WEIGHTS       = 'yolov8m-pose.pt'

# API key de Anthropic — pégala aquí o cárgala desde un fichero en Drive
# Opción A (directa):   ANTHROPIC_API_KEY = 'sk-ant-...'
# Opción B (desde fichero en Drive, recomendado):
ANTHROPIC_API_KEY = open('/content/drive/MyDrive/TFM/secrets/anthropic_key.txt').read().strip()

# Inferencia
IMG_SIZE     = 640
CONF_WEAPON  = 0.25
IOU_NMS      = 0.7
CONF_POSE    = 0.5
FRAME_STRIDE = 1
MAX_SECONDS  = 15

# Pipeline Modelo B
DETECTION_THRESHOLD = 5

# Filtro temporal de pose
KP_CONF_THR      = 0.3
AIMING_ANGLE_THR = 160
POSE_WINDOW      = 10
AIMING_RUN_THR   = 3

# LLM — cuántos clips analizar (None = todos)
# Empieza con pocos para validar el prompt antes de lanzar el dataset completo
MAX_CLIPS_POS = 3
MAX_CLIPS_NEG = 3

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('OK Config')

OK Config


In [5]:
# ============================================================
# KEYPOINTS COCO + HELPERS GEOMÉTRICOS
# (idénticos al notebook 12)
# ============================================================

L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW,    R_ELBOW    = 7, 8
L_WRIST,    R_WRIST    = 9, 10

def angle_between(a, b, c):
    ba = (a[0]-b[0], a[1]-b[1])
    bc = (c[0]-b[0], c[1]-b[1])
    dot  = ba[0]*bc[0] + ba[1]*bc[1]
    norm = math.sqrt(ba[0]**2+ba[1]**2) * math.sqrt(bc[0]**2+bc[1]**2)
    if norm < 1e-6:
        return None
    return math.degrees(math.acos(max(-1.0, min(1.0, dot/norm))))

def get_arm_angles(kps, confs):
    results = {'left': None, 'right': None}
    for side, (sh, el, wr) in [('left',  (L_SHOULDER, L_ELBOW, L_WRIST)),
                                 ('right', (R_SHOULDER, R_ELBOW, R_WRIST))]:
        if all(confs[i] >= KP_CONF_THR for i in [sh, el, wr]):
            results[side] = angle_between(tuple(kps[sh]), tuple(kps[el]), tuple(kps[wr]))
    return results

def classify_pose(arm_angles):
    valid = {k: v for k, v in arm_angles.items() if v is not None}
    if not valid:
        return 'NEUTRAL'
    if any(v >= AIMING_ANGLE_THR for v in valid.values()):
        return 'AIMING'
    return 'HOLDING'

def bbox_center(box):
    return ((box[0]+box[2])/2, (box[1]+box[3])/2)

def find_nearest_person(weapon_box, person_boxes):
    if not person_boxes:
        return None
    wc = bbox_center(weapon_box)
    return int(np.argmin([math.dist(wc, bbox_center(pb)) for pb in person_boxes]))

print('OK Helpers')

OK Helpers


In [6]:
# Copiar pesos a /content/ para evitar desconexiones de Drive
LOCAL_WEAPON_WEIGHTS = '/content/weapon_best.pt'
if not os.path.exists(LOCAL_WEAPON_WEIGHTS):
    print('Copiando pesos desde Drive...')
    shutil.copy2(WEAPON_DET_WEIGHTS, LOCAL_WEAPON_WEIGHTS)

weapon_model = YOLO(LOCAL_WEAPON_WEIGHTS)
pose_model   = YOLO(POSE_WEIGHTS)
llm_client   = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print('OK Modelos + cliente LLM listos')

Copiando pesos desde Drive...
OK Modelos + cliente LLM listos


In [7]:
# ============================================================
# PROCESADOR DE CLIP
# Acumula métricas completas para pasarlas al LLM
# ============================================================

def process_clip_for_llm(video_path):
    """
    Procesa el clip y devuelve un dict con todas las métricas
    necesarias para construir el prompt del LLM.
    """
    local_in = '/content/tmp_llm.mp4'
    shutil.copy2(str(video_path), local_in)

    cap      = cv2.VideoCapture(local_in)
    fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_f    = min(n_frames, int(MAX_SECONDS * fps))

    frames_with_weapon = 0
    frames_with_pose   = 0
    threat_confirmed   = False
    pose_window        = deque(maxlen=POSE_WINDOW)

    # Acumuladores para el resumen
    label_counts  = defaultdict(int)   # AIMING / HOLDING / NEUTRAL
    angle_L_list  = []
    angle_R_list  = []
    # Secuencia temporal comprimida: bloques consecutivos del mismo label
    pose_sequence = []   # lista de labels en orden cronológico
    weapon_conf_list = []

    frame_i = 0
    while frame_i < max_f:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_i % FRAME_STRIDE != 0:
            frame_i += 1
            continue

        # Detección de armas
        wr = weapon_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                                   iou=IOU_NMS, verbose=False, device='cuda')[0]
        weapon_boxes = []
        if wr.boxes is not None and len(wr.boxes) > 0:
            for b in wr.boxes:
                x1,y1,x2,y2 = map(float, b.xyxy[0])
                conf = float(b.conf[0])
                weapon_boxes.append((x1,y1,x2,y2,conf))
                weapon_conf_list.append(conf)

        if weapon_boxes:
            frames_with_weapon += 1

            # Estimación de pose
            pr = pose_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_POSE,
                                     verbose=False, device='cuda')[0]
            pose_label = 'NEUTRAL'
            person_boxes_raw = []

            if pr.keypoints is not None and len(pr.keypoints) > 0:
                kps_data  = pr.keypoints.xy.cpu().numpy()
                conf_data = pr.keypoints.conf.cpu().numpy()
                if pr.boxes is not None:
                    for b in pr.boxes:
                        person_boxes_raw.append(list(map(float, b.xyxy[0])))
                nearest = find_nearest_person(weapon_boxes[0][:4], person_boxes_raw)
                if nearest is not None and nearest < len(kps_data):
                    angles = get_arm_angles(kps_data[nearest], conf_data[nearest])
                    pose_label = classify_pose(angles)
                    frames_with_pose += 1
                    if angles['left']  is not None: angle_L_list.append(angles['left'])
                    if angles['right'] is not None: angle_R_list.append(angles['right'])

            label_counts[pose_label] += 1
            pose_sequence.append(pose_label)
            pose_window.append(pose_label)

            n_aiming = sum(1 for p in pose_window if p == 'AIMING')
            if n_aiming >= AIMING_RUN_THR:
                threat_confirmed = True

        frame_i += 1

    cap.release()
    os.remove(local_in)

    # Comprimir secuencia temporal en bloques (run-length encoding)
    rle = []
    if pose_sequence:
        current, count = pose_sequence[0], 1
        for label in pose_sequence[1:]:
            if label == current:
                count += 1
            else:
                rle.append(f'{current}x{count}')
                current, count = label, 1
        rle.append(f'{current}x{count}')

    total_pose_frames = sum(label_counts.values())

    return {
        'total_frames_analyzed':  max_f,
        'fps':                    round(fps, 1),
        'frames_with_weapon':     frames_with_weapon,
        'weapon_pct':             round(frames_with_weapon / max_f * 100, 1) if max_f > 0 else 0,
        'mean_weapon_conf':       round(float(np.mean(weapon_conf_list)), 3) if weapon_conf_list else 0,
        'frames_with_pose':       frames_with_pose,
        'label_counts':           dict(label_counts),
        'aiming_pct':             round(label_counts['AIMING'] / total_pose_frames * 100, 1) if total_pose_frames > 0 else 0,
        'mean_angle_L':           round(float(np.mean(angle_L_list)), 1) if angle_L_list else None,
        'mean_angle_R':           round(float(np.mean(angle_R_list)), 1) if angle_R_list else None,
        'pose_sequence_rle':      rle,
        'pred_modelB':            1 if frames_with_weapon >= DETECTION_THRESHOLD else 0,
        'threat_confirmed':       threat_confirmed,
    }

print('OK process_clip_for_llm definida')

OK process_clip_for_llm definida


In [8]:
# ============================================================
# LLM: CONSTRUCCIÓN DE PROMPT Y LLAMADA A CLAUDE
# ============================================================

SYSTEM_PROMPT = """You are an expert AI security analyst specialized in video surveillance.
You receive structured data from a multi-stage weapon detection pipeline:
  Stage 1: Object detector (YOLOv8) identifies firearms in video frames
  Stage 2: Human Pose Estimation (YOLOv8-pose) classifies body posture as
            AIMING (arm extended, elbow angle >= 160 deg),
            HOLDING (arm bent, weapon detected but not raised),
            or NEUTRAL (no reliable keypoints detected)
  Stage 3: Temporal filter flags threat if >= 3 AIMING frames occur within a 10-frame window

Your task:
  1. Analyze the provided clip summary
  2. Reason about the TEMPORAL PATTERN of poses (not just counts)
  3. Assign a threat level: HIGH / MEDIUM / LOW / NONE
  4. Write a concise narrative interpretation (2-4 sentences) suitable for a security report
  5. Note any limitations or uncertainties in the analysis

Respond ONLY with valid JSON in this exact schema:
{
  "threat_level": "HIGH|MEDIUM|LOW|NONE",
  "narrative": "...",
  "temporal_reasoning": "...",
  "limitations": "...",
  "confidence": 0.0
}"""


def build_user_prompt(clip_id, metrics):
    """
    Construye el mensaje de usuario para el LLM a partir
    de las métricas acumuladas del clip.
    """
    lc = metrics['label_counts']
    total_pose = sum(lc.values())

    pose_dist = ''
    for label in ['AIMING', 'HOLDING', 'NEUTRAL']:
        n = lc.get(label, 0)
        pct = n / total_pose * 100 if total_pose > 0 else 0
        pose_dist += f'  {label}: {n} frames ({pct:.1f}%)\n'

    angle_str = ''
    if metrics['mean_angle_L'] is not None:
        angle_str += f'  Mean left arm angle : {metrics["mean_angle_L"]} deg\n'
    if metrics['mean_angle_R'] is not None:
        angle_str += f'  Mean right arm angle: {metrics["mean_angle_R"]} deg\n'
    if not angle_str:
        angle_str = '  No reliable arm keypoints detected\n'

    # Secuencia temporal como string legible
    seq_str = ' -> '.join(metrics['pose_sequence_rle'][:20])
    if len(metrics['pose_sequence_rle']) > 20:
        seq_str += ' -> [...]'

    prompt = f"""CLIP ANALYSIS REPORT
Clip ID       : {clip_id}
Duration      : {metrics['total_frames_analyzed']} frames at {metrics['fps']} fps
               ({metrics['total_frames_analyzed'] / metrics['fps']:.1f} seconds analyzed)

--- WEAPON DETECTION (Stage 1) ---
Frames with weapon detected : {metrics['frames_with_weapon']} ({metrics['weapon_pct']}% of clip)
Mean detection confidence   : {metrics['mean_weapon_conf']}
Model B prediction          : {'POSITIVE (weapon present)' if metrics['pred_modelB'] else 'NEGATIVE'}

--- POSE ANALYSIS (Stage 2) ---
Frames with pose estimated  : {metrics['frames_with_pose']}
Pose distribution:
{pose_dist}
--- ARM ANGLES ---
{angle_str}
--- TEMPORAL FILTER (Stage 3) ---
Parameters    : window={POSE_WINDOW} frames, min_aiming={AIMING_RUN_THR}
Threat flag   : {'CONFIRMED' if metrics['threat_confirmed'] else 'NOT CONFIRMED'}

--- TEMPORAL POSE SEQUENCE (run-length encoded) ---
{seq_str}

Based on this data, provide your threat assessment."""
    return prompt


def call_llm(clip_id, metrics, max_retries=2):
    """
    Llama a Claude con el resumen del clip.
    Retorna el dict parseado de la respuesta JSON.
    """
    user_prompt = build_user_prompt(clip_id, metrics)

    for attempt in range(max_retries + 1):
        try:
            response = llm_client.messages.create(
                model='claude-sonnet-4-20250514',
                max_tokens=512,
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': user_prompt}]
            )
            raw = response.content[0].text.strip()
            # Limpiar posibles backticks de markdown
            raw = raw.replace('```json', '').replace('```', '').strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            if attempt < max_retries:
                print(f'    [retry {attempt+1}] JSON parse error: {e}')
            else:
                return {'threat_level': 'ERROR', 'narrative': str(e),
                        'temporal_reasoning': '', 'limitations': '', 'confidence': 0.0}
        except Exception as e:
            return {'threat_level': 'ERROR', 'narrative': str(e),
                    'temporal_reasoning': '', 'limitations': '', 'confidence': 0.0}


print('OK LLM helpers definidos')
print(f'System prompt ({len(SYSTEM_PROMPT)} chars):')
print(SYSTEM_PROMPT[:200] + '...')

OK LLM helpers definidos
System prompt (1083 chars):
You are an expert AI security analyst specialized in video surveillance.
You receive structured data from a multi-stage weapon detection pipeline:
  Stage 1: Object detector (YOLOv8) identifies firear...


In [9]:
# ============================================================
# VALIDACIÓN: probar prompt con un único clip positivo
# Ejecuta esta celda primero para verificar la respuesta del LLM
# antes de lanzar el procesamiento completo
# ============================================================

pos_paths_all = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]
test_path = next(vp for vp in pos_paths_all if vp.exists())
test_id   = test_path.parent.name

print(f'Procesando clip de prueba: {test_id}')
test_metrics = process_clip_for_llm(test_path)

print('\n--- Métricas calculadas ---')
for k, v in test_metrics.items():
    if k != 'pose_sequence_rle':
        print(f'  {k}: {v}')
print(f'  pose_sequence_rle: {test_metrics["pose_sequence_rle"][:10]}...')

print('\n--- Prompt que se enviará al LLM ---')
print(build_user_prompt(test_id, test_metrics))

print('\n--- Respuesta del LLM ---')
test_response = call_llm(test_id, test_metrics)
print(json.dumps(test_response, indent=2, ensure_ascii=False))

Procesando clip de prueba: PAH1_C1_P1_V1_HB_3

--- Métricas calculadas ---
  total_frames_analyzed: 150
  fps: 25.0
  frames_with_weapon: 25
  weapon_pct: 16.7
  mean_weapon_conf: 0.569
  frames_with_pose: 25
  label_counts: {'AIMING': 25}
  aiming_pct: 100.0
  mean_angle_L: 161.2
  mean_angle_R: 166.7
  pred_modelB: 1
  threat_confirmed: True
  pose_sequence_rle: ['AIMINGx25']...

--- Prompt que se enviará al LLM ---
CLIP ANALYSIS REPORT
Clip ID       : PAH1_C1_P1_V1_HB_3
Duration      : 150 frames at 25.0 fps
               (6.0 seconds analyzed)

--- WEAPON DETECTION (Stage 1) ---
Frames with weapon detected : 25 (16.7% of clip)
Mean detection confidence   : 0.569
Model B prediction          : POSITIVE (weapon present)

--- POSE ANALYSIS (Stage 2) ---
Frames with pose estimated  : 25
Pose distribution:
  AIMING: 25 frames (100.0%)
  HOLDING: 0 frames (0.0%)
  NEUTRAL: 0 frames (0.0%)

--- ARM ANGLES ---
  Mean left arm angle : 161.2 deg
  Mean right arm angle: 166.7 deg

--- TEMPORA

In [10]:
# ============================================================
# EVALUACIÓN COMPLETA
# Procesa MAX_CLIPS_POS positivos + MAX_CLIPS_NEG negativos
# ============================================================
print('=' * 60)
print('EVALUACIÓN COMPLETA CON LLM')
print('=' * 60)

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]
neg_paths = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]

if MAX_CLIPS_POS: pos_paths = pos_paths[:MAX_CLIPS_POS]
if MAX_CLIPS_NEG: neg_paths = neg_paths[:MAX_CLIPS_NEG]

all_results = []

def process_and_analyze(paths, true_label, label_str):
    print(f'\n--- {label_str} ({len(paths)} clips) ---')
    for vp in paths:
        if not vp.exists():
            continue
        clip_id  = vp.parent.name
        category = clip_id.split('_')[0] if true_label == 0 else 'POS'

        # Stage 1+2+3: visión
        metrics = process_clip_for_llm(vp)

        # Stage 4: LLM (solo si el Modelo B disparó)
        llm_result = None
        if metrics['pred_modelB'] == 1:
            llm_result = call_llm(clip_id, metrics)

        result = {
            'clip_id':    clip_id,
            'true':       true_label,
            'category':   category,
            **metrics,
            'llm_threat_level':      llm_result['threat_level']      if llm_result else 'N/A',
            'llm_confidence':        llm_result['confidence']         if llm_result else None,
            'llm_narrative':         llm_result['narrative']          if llm_result else '',
            'llm_temporal_reasoning':llm_result['temporal_reasoning'] if llm_result else '',
            'llm_limitations':       llm_result['limitations']        if llm_result else '',
        }
        all_results.append(result)

        # Imprimir resumen
        hpe_flag = 'THREAT' if metrics['threat_confirmed'] else 'no-threat'
        llm_flag = llm_result['threat_level'] if llm_result else 'skipped'
        print(f'  {clip_id:<35}  ModeloB={metrics["pred_modelB"]}  '
              f'HPE={hpe_flag}  LLM={llm_flag}')
        if llm_result and llm_result['threat_level'] not in ['NONE', 'N/A', 'ERROR']:
            print(f'    Narrative: {llm_result["narrative"][:120]}')


process_and_analyze(pos_paths, true_label=1, label_str='POSITIVOS')
process_and_analyze(neg_paths, true_label=0, label_str='NEGATIVOS')

print(f'\nTotal clips procesados: {len(all_results)}')

EVALUACIÓN COMPLETA CON LLM

--- POSITIVOS (3 clips) ---
  PAH1_C1_P1_V1_HB_3                   ModeloB=1  HPE=THREAT  LLM=HIGH
    Narrative: Subject maintains a consistent aiming posture throughout the entire 6-second observation period, with both arms extended
  PAH1_C1_P1_V1_HB_4                   ModeloB=1  HPE=THREAT  LLM=HIGH
    Narrative: Individual detected holding a firearm for the majority of the 7-second surveillance period, with multiple deliberate aim
  PAH1_C1_P2_V1_HB_1                   ModeloB=1  HPE=THREAT  LLM=HIGH
    Narrative: Analysis confirms a credible threat scenario with sustained weapon handling and multiple distinct aiming sequences. The 

--- NEGATIVOS (3 clips) ---
  N10_C1_P1_V1_HB_1                    ModeloB=0  HPE=no-threat  LLM=skipped
  N10_C1_P2_V1_HB_1                    ModeloB=0  HPE=no-threat  LLM=skipped
  N10_C1_P4_V1_HB_2                    ModeloB=1  HPE=THREAT  LLM=HIGH
    Narrative: Security footage shows a confirmed weapon threat with

In [11]:
# ============================================================
# RESUMEN: distribución de niveles de alerta del LLM
# ============================================================
print('=' * 60)
print('RESUMEN — Distribución de threat_level por LLM')
print('=' * 60)

# Solo clips donde el LLM fue invocado (ModeloB=1)
llm_invoked = [r for r in all_results if r['llm_threat_level'] != 'N/A']

print(f'\n  Clips donde LLM fue invocado: {len(llm_invoked)}')
print(f'  (solo cuando Modelo B = POSITIVE)\n')

# Por true label
for true_label, label_str in [(1, 'POSITIVOS'), (0, 'NEGATIVOS')]:
    subset = [r for r in llm_invoked if r['true'] == true_label]
    if not subset:
        continue
    level_counts = defaultdict(int)
    for r in subset:
        level_counts[r['llm_threat_level']] += 1
    print(f'  {label_str} (n={len(subset)}):')
    for level in ['HIGH', 'MEDIUM', 'LOW', 'NONE', 'ERROR']:
        n = level_counts.get(level, 0)
        if n > 0:
            print(f'    {level:<8}: {n} clips ({n/len(subset)*100:.0f}%)')

# Comparativa: HPE vs LLM sobre los clips donde ModeloB=1
print(f'\n  Comparativa HPE vs LLM (sobre clips ModeloB=1):')
print(f'  {"":<35} {"HPE":>8} {"LLM":>10}')
print('  ' + '-'*55)
HIGH_MEDIUM = {'HIGH', 'MEDIUM'}
for r in llm_invoked:
    hpe = 'THREAT' if r['threat_confirmed'] else 'no-threat'
    llm = r['llm_threat_level']
    agree = '✓' if (r['threat_confirmed'] == (llm in HIGH_MEDIUM)) else '✗'
    print(f"  {r['clip_id']:<35} {hpe:>8} {llm:>10}  {agree}  (true={'POS' if r['true']==1 else 'NEG'})")

RESUMEN — Distribución de threat_level por LLM

  Clips donde LLM fue invocado: 4
  (solo cuando Modelo B = POSITIVE)

  POSITIVOS (n=3):
    HIGH    : 3 clips (100%)
  NEGATIVOS (n=1):
    HIGH    : 1 clips (100%)

  Comparativa HPE vs LLM (sobre clips ModeloB=1):
                                           HPE        LLM
  -------------------------------------------------------
  PAH1_C1_P1_V1_HB_3                    THREAT       HIGH  ✓  (true=POS)
  PAH1_C1_P1_V1_HB_4                    THREAT       HIGH  ✓  (true=POS)
  PAH1_C1_P2_V1_HB_1                    THREAT       HIGH  ✓  (true=POS)
  N10_C1_P4_V1_HB_2                     THREAT       HIGH  ✓  (true=NEG)


In [12]:
# ============================================================
# GUARDAR RESULTADOS
# ============================================================

# CSV completo
csv_path = Path(OUT_DIR) / 'llm_intent_results.csv'
if all_results:
    fieldnames = [
        'clip_id', 'true', 'category',
        'frames_with_weapon', 'weapon_pct', 'mean_weapon_conf',
        'aiming_pct', 'pred_modelB', 'threat_confirmed',
        'llm_threat_level', 'llm_confidence',
        'llm_narrative', 'llm_temporal_reasoning', 'llm_limitations'
    ]
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(all_results)
    print(f'CSV guardado en: {csv_path}')

# Narrativas completas en texto plano (para la memoria del TFM)
narratives_path = Path(OUT_DIR) / 'llm_narratives.txt'
with open(narratives_path, 'w', encoding='utf-8') as f:
    for r in all_results:
        if r['llm_threat_level'] == 'N/A':
            continue
        f.write(f"{'='*60}\n")
        f.write(f"Clip     : {r['clip_id']}\n")
        f.write(f"True     : {'POSITIVO' if r['true']==1 else 'NEGATIVO'}\n")
        f.write(f"ModeloB  : {r['pred_modelB']}  HPE: {'THREAT' if r['threat_confirmed'] else 'no-threat'}\n")
        f.write(f"LLM level: {r['llm_threat_level']}  (conf={r['llm_confidence']})\n\n")
        f.write(f"Narrative:\n{r['llm_narrative']}\n\n")
        f.write(f"Temporal reasoning:\n{r['llm_temporal_reasoning']}\n\n")
        f.write(f"Limitations:\n{r['llm_limitations']}\n\n")
print(f'Narrativas guardadas en: {narratives_path}')
print('OK Evaluacion con LLM completa.')

CSV guardado en: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/llm_intent/llm_intent_results.csv
Narrativas guardadas en: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/llm_intent/llm_narratives.txt
OK Evaluacion con LLM completa.
